In [2]:
# 変化量の上位95%だけを抽出
import pandas as pd
import os

# 入力ファイル
input_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/vad_dis.csv"

# 出力ファイル
output_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/results/Δint-95.csv"

# 出力先が無ければ新規作成
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# CSV読み込み
df = pd.read_csv(input_csv)

filter_95 = df[df["delta_int_kushinada"] > 4.06]

result = filter_95[
    [
        "filename",
        "text",
        "valence",
        "arousal",
        "dominance",
        "intensity_pred_kushinada",
        "delta_valence",
        "delta_arousal",
        "delta_dominance",
        "delta_int_kushinada",
    ]
]

# 保存
result.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"保存完了: {output_csv}")
print(f"発話数: {len(result)}")

保存完了: /home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/results/Δint-95.csv
発話数: 14


In [3]:
df[df["delta_int_kushinada"] > 4.06]

,filename,text,valence,arousal,dominance,intensity_pred_kushinada,delta_valence,delta_arousal,delta_dominance,delta_int_kushinada
25,0050_L.wav,(F ええ)(F ええ),0.39,0.35,0.35,6.53,0.01,0.17,0.17,4.12
48,0094_L.wav,(F うーん),0.43,0.37,0.36,4.89,0.03,0.18,0.19,4.24
63,0130_L.wav,(D2 お),0.14,0.24,0.20,0.85,0.26,0.04,0.01,5.13
76,0159_L.wav,じゃ最近のはやりじゃないですか,0.37,0.62,0.60,-0.40,0.05,0.22,0.18,4.86
114,0230_L.wav,(F はい)(F はい)(F はい)(F はい),0.81,0.65,0.60,0.77,0.46,0.35,0.34,4.46
118,0238_L.wav,(F うーん),0.37,0.17,0.18,4.56,0.10,0.04,0.02,4.25
128,0256_L.wav,<笑>,0.90,0.80,0.76,4.12,0.44,0.65,0.51,4.06
171,0339_L.wav,そうですよね,0.40,0.08,0.16,6.05,0.11,0.43,0.33,4.36
178,0357_L.wav,(F うーん),0.36,0.00,0.09,4.52,0.05,0.56,0.46,4.15
187,0377_L.wav,また大変でしょう小学校入ったら入ったでねきっとね,0.18,0.29,0.39,0.85,0.67,0.43,0.26,5.05


In [4]:
# パーセンタイル95%のLの発話
import pandas as pd
import os
import shutil

# ==========================
# パス
# ==========================
l_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/vad_dis.csv"
wav_dir = "/home/mitani/CSJ-emo-int_bunseki/ALL-WAV"
output_root = "/home/mitani/CSJ-emo-int_bunseki/0718/L_bunseki/results/eval-int"

os.makedirs(output_root, exist_ok=True)

# ==========================
# CSV読込（Lのみ）
# ==========================
df_l = pd.read_csv(l_csv)

# 発話番号（CSVのインデックスではなく、0015_L.wav の「15」を指定）
change_idx = [
    50, 94, 130, 159, 230, 238, 256, 339, 357, 377, 409, 434, 479, 482
]

context = 3

# ==========================
# 抽出
# ==========================
for utt_no in change_idx:

    # 発話番号からファイル名を作成
    target_filename = f"{utt_no:04d}_L.wav"

    # filenameから該当行を検索
    matched = df_l[df_l["filename"] == target_filename]

    if matched.empty:
        print(f"{target_filename} がCSV内に見つかりません")
        continue

    # DataFrame内でのインデックス取得
    idx = matched.index[0]

    print(f"発話番号 {utt_no} -> DataFrame index {idx}")

    # 前後3発話（Lのみ）
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Utterance number : {utt_no}\n")
        f.write(f"DataFrame index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 音声コピー
    for _, row in context_df.iterrows():

        src = os.path.join(wav_dir, row["filename"])

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # ターゲット音声にはTARGET_を付ける
        if row["filename"] == target_filename:
            dst_name = "TARGET_" + row["filename"]
        else:
            dst_name = row["filename"]

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

発話番号 50 -> DataFrame index 25
発話番号 94 -> DataFrame index 48
発話番号 130 -> DataFrame index 63
発話番号 159 -> DataFrame index 76
発話番号 230 -> DataFrame index 114
発話番号 238 -> DataFrame index 118
発話番号 256 -> DataFrame index 128
発話番号 339 -> DataFrame index 171
発話番号 357 -> DataFrame index 178
発話番号 377 -> DataFrame index 187
発話番号 409 -> DataFrame index 201
発話番号 434 -> DataFrame index 215
発話番号 479 -> DataFrame index 238
発話番号 482 -> DataFrame index 239
完了
